In [43]:
import pandas as pd
import numpy as np

In [44]:
pd.set_option("display.max_columns", None)
df = pd.read_csv("../data/raw/dataset-ventas-2025-2026.csv")

C:\Users\Admin\AppData\Local\Temp\ipykernel_17484\676536017.py:2: DtypeWarning: Columns (0: precio) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/dataset-ventas-2025-2026.csv")


In [45]:
df.head()

,idtabla,fechaventa,agente,supervisor,SUCURSAL,REGION,paquete_vendido,id_paquete,precio,TipoServicio
0,0,2025-09-03 10:38:26.000,5526,5380,Tlaquepaque,OCCIDENTE,NETFLIX PREMIUM 329 329,95,329,NETFLIX
1,46698,2025-08-22 13:02:31.000,5872,5720,Cd. Morelia,TCO,PLAN ALTAN 100 100,102,100,MEGAMOVIL
2,60365,2025-08-27 16:01:36.000,5826,5468,Tonalá,OCCIDENTE,DISNEY STANDARD CON ANUNCIOS 60 60,112,60,DISNEY
3,65956,2025-09-15 15:46:42.000,5945,5468,Tlajomulco,OCCIDENTE,DISNEY STANDARD CON ANUNCIOS 60 60,112,60,DISNEY
4,90262,2025-09-12 13:42:20.000,4125,3942,Zacapu,TCO,PLAN AT&T 100 100,105,100,MEGAMOVIL


In [46]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 126827 entries, 0 to 126826
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   idtabla          126827 non-null  int64 
 1   fechaventa       126827 non-null  str   
 2   agente           126827 non-null  int64 
 3   supervisor       126827 non-null  int64 
 4   SUCURSAL         126827 non-null  str   
 5   REGION           126827 non-null  str   
 6   paquete_vendido  126814 non-null  str   
 7   id_paquete       126827 non-null  int64 
 8   precio           126806 non-null  object
 9   TipoServicio     126780 non-null  str   
dtypes: int64(4), object(1), str(5)
memory usage: 9.7+ MB


In [47]:
df["fechaventa"] = pd.to_datetime(df["fechaventa"])
df = df.drop_duplicates()

In [48]:
# Normalizar regiones
df["REGION"] = (
    df["REGION"].str.strip().str.upper()
)

In [49]:
# Ver null en tipo servicio
df["TipoServicio"].isna().sum()
df_null = df[df["TipoServicio"].isna()]
df_null.head()

,idtabla,fechaventa,agente,supervisor,SUCURSAL,REGION,paquete_vendido,id_paquete,precio,TipoServicio
406,176912,2025-06-03 09:46:59,5805,5468,Zapopan,OCCIDENTE,PLAN ALTAN 150 150,78,50,NaN
6299,182875,2025-06-25 12:22:56,5805,5468,Tijuana,PACIFICO,PLAN ALTAN 100 100,102,00,NaN
36509,213798,2025-09-25 10:00:07,5807,5468,Cd. Colima,OCCIDENTE,PLAN AT&T 100 100,105,100,NaN
41579,218969,2025-10-06 12:22:40,5817,3942,Laguna Larga,OCCIDENTE,PLAN AT&T 100 100,105,100,NaN
41580,218970,2025-10-06 12:22:40,5817,3942,Laguna Larga,OCCIDENTE,PLAN AT&T 100 100,105,100,NaN


In [50]:
# Diccionario con id_paquete
lookup = (
    df.dropna(subset=["TipoServicio"]).groupby("id_paquete")["TipoServicio"].agg(lambda x: x.mode().iloc[0])
)

In [51]:
df["TipoServicio"] = df["TipoServicio"].fillna(
    df["id_paquete"].map(lookup)
)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 126827 entries, 0 to 126826
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   idtabla          126827 non-null  int64         
 1   fechaventa       126827 non-null  datetime64[us]
 2   agente           126827 non-null  int64         
 3   supervisor       126827 non-null  int64         
 4   SUCURSAL         126827 non-null  str           
 5   REGION           126827 non-null  str           
 6   paquete_vendido  126814 non-null  str           
 7   id_paquete       126827 non-null  int64         
 8   precio           126806 non-null  object        
 9   TipoServicio     126827 non-null  str           
dtypes: datetime64[us](1), int64(4), object(1), str(4)
memory usage: 9.7+ MB


In [52]:
# Diccionario para paquete_vendido
lookup_paq = (
    df.dropna(subset=["paquete_vendido"]).groupby("id_paquete")["paquete_vendido"].agg(lambda x: x.mode().iloc[0])
)

In [53]:
df["paquete_vendido"] = df["paquete_vendido"].fillna(
    df["id_paquete"].map(lookup_paq)
)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 126827 entries, 0 to 126826
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   idtabla          126827 non-null  int64         
 1   fechaventa       126827 non-null  datetime64[us]
 2   agente           126827 non-null  int64         
 3   supervisor       126827 non-null  int64         
 4   SUCURSAL         126827 non-null  str           
 5   REGION           126827 non-null  str           
 6   paquete_vendido  126827 non-null  str           
 7   id_paquete       126827 non-null  int64         
 8   precio           126806 non-null  object        
 9   TipoServicio     126827 non-null  str           
dtypes: datetime64[us](1), int64(4), object(1), str(4)
memory usage: 9.7+ MB


In [54]:
# diccionario para precio
df_null_precio = df[df["precio"].isna()]
df_null_precio.head()

,idtabla,fechaventa,agente,supervisor,SUCURSAL,REGION,paquete_vendido,id_paquete,precio,TipoServicio
17105,193935,2025-08-07 15:36:26,5971,4714,Miguel Aleman,PACIFICO,PLAN AT&T 100 100,105,NaN,ADULT PACK
17106,193936,2025-08-07 15:36:46,5971,4714,Miguel Aleman,PACIFICO,PLAN AT&T 100 100,105,NaN,ADULT PACK
17108,193938,2025-08-07 15:37:12,5971,4714,Miguel Aleman,PACIFICO,PLAN AT&T 100 100,105,NaN,ADULT PACK
18717,195595,2025-08-13 16:35:39,5980,5720,Cd. Guadalajara,OCCIDENTE,PLAN ALTAN 100 100,102,NaN,ADULT PACK
18718,195596,2025-08-13 16:36:01,5980,5720,Cd. Guadalajara,OCCIDENTE,PLAN ALTAN 100 100,102,NaN,ADULT PACK


In [55]:
lookup_pre = (
    df.dropna(subset=["precio"]).groupby("id_paquete")["precio"].agg(lambda x: x.mode().iloc[0])
)

In [56]:
df["precio"] = df["precio"].fillna(
    df["id_paquete"].map(lookup_pre)
)
df["precio"] = df["precio"].astype(str).str.replace(r"[A-Za-z]", "", regex=True)
df["precio"] = df["precio"].str.strip()
df["precio"] = pd.to_numeric(df["precio"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 126827 entries, 0 to 126826
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   idtabla          126827 non-null  int64         
 1   fechaventa       126827 non-null  datetime64[us]
 2   agente           126827 non-null  int64         
 3   supervisor       126827 non-null  int64         
 4   SUCURSAL         126827 non-null  str           
 5   REGION           126827 non-null  str           
 6   paquete_vendido  126827 non-null  str           
 7   id_paquete       126827 non-null  int64         
 8   precio           126827 non-null  float64       
 9   TipoServicio     126827 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(4), str(4)
memory usage: 9.7 MB


In [57]:
df.to_csv("../data/processed/dataset-ventas-2025-2026-clean.csv", index=False)